In [ ]:
import os
from pathlib import Path
import re

import pandas as pd

from kibad_llm.config import PROJ_ROOT
from kibad_llm.utils.job_return import group_by, load_subdirs

# swith to project root to use same paths as in commands
os.chdir(PROJ_ROOT)
# set wider column width for displaying pandas data frames
pd.set_option("max_colwidth", 400)
# show all columns
pd.set_option("display.max_columns", None)

In [ ]:
BASE_DIR = Path("data/prediction_results/logs/159_core_schema_baseline_multirun")
METRICS_DIR = BASE_DIR / "evaluate"
ERRORS_DIR = BASE_DIR / "evaluate_errors"
FILL_NA = {"overrides.extractor/llm": "gpt_oss_20b"}

# don't load this data to declutter the data frames
EXCLUDE_KEYS = [
    "prediction.job_return_value.output_file",
    "overrides.paths.save_dir",
    "overrides.pdf_directory",
]

In [ ]:
metrics_df = pd.DataFrame.from_records(
    load_subdirs(METRICS_DIR, strip_id_keys=True, flatten=True, exclude_keys=EXCLUDE_KEYS)
).fillna(FILL_NA)
metrics_df

In [ ]:
group_by(metrics_df, by="overrides.extractor/llm")

In [ ]:
def plot_mean(
    df: pd.DataFrame,
    by: str | list[str],
    include_cols_regex: str,
    with_std_as_err: bool = False,
    xlabel: str | None = None,
) -> None:
    if isinstance(by, str):
        by = [by]
    cols = [col for col in df.columns if re.search(include_cols_regex, col)] + by

    # create grouped dataframe with mean (and std), indexed by 'by' columns
    y = group_by(df[cols], by=by, numeric_agg_func="mean")
    yerr = group_by(df[cols], by=by, numeric_agg_func="std") if with_std_as_err else None

    # bar plot with mean and std as error bars
    ax = y.plot(kind="bar", yerr=yerr, capsize=4)  # , rot=45)

    ax.set_xlabel("\n".join(by))
    ax.set_ylabel(xlabel or f"mean: {include_cols_regex}")
    ax.legend(title="metric", bbox_to_anchor=(1.02, 1), loc="upper left")

    # "tight_layout"
    # ax.figure.tight_layout()

    # In notebooks this is usually enough.
    # In scripts, this may or may not pop up a window depending on backend:
    # ax.figure.show()

# metrics

In [ ]:
plot_mean(
    metrics_df, by="overrides.extractor/llm", include_cols_regex=r"[^G]\.f1", with_std_as_err=True
)  # , xlabel="mean f1")

# support

In [ ]:
cols_to_plot = [col for col in metrics_df.columns if "support" in col and "ALL" not in col]
# get the first row, but as dataframe so that we have color-coded bars and a legend
ax = metrics_df[cols_to_plot].iloc[0:1].plot(kind="bar")
ax.legend(title="type", bbox_to_anchor=(1.02, 1), loc="upper left")
# ax.figure.tight_layout()

In [ ]:
errors_df = pd.DataFrame.from_records(
    load_subdirs(ERRORS_DIR, strip_id_keys=True, flatten=True, exclude_keys=EXCLUDE_KEYS)
).fillna(FILL_NA)
# we analyse error counts, so aggregate by summing them up
errors_df_avg = group_by(errors_df, by="overrides.extractor/llm", numeric_agg_func="sum")
errors_df_avg

# errors

In [ ]:
# cols_to_plot = errors_df_avg.select_dtypes(include="number").columns
cols_to_plot = [col for col in errors_df_avg.columns if "error" in str(col) or "Error" in str(col)]
ax = errors_df_avg.plot.bar(stacked=True, y=cols_to_plot)
ax.legend(title="type", bbox_to_anchor=(1.02, 1), loc="upper left")
# ax.figure.tight_layout()